# NB15 — Results Finalization (masterclass)

Regenerates **every paper table and journal-quality figure from raw result files** into
`04_outputs/finalized_outputs/{tables,figures,diagrams}`. Supersedes the old NB15 + NB16.

**Non-destructive:** reads from `04_outputs/...` and the raw splits; only ever *writes* inside
`finalized_outputs/`. Delete the old `finalized_outputs/` first (see the accompanying cleanup
commands) so this rebuilds cleanly.

Pipeline: setup → load → figure engine → generate figures → optional CPU cross-corpus classical
baseline → copy diagrams → whitelist paper tables → headline table + manifest.

> **Proposed model = BanglaBERT full fine-tuning + label-smoothed dual-pool head.**
> Adversarial training (FGM/AWP) and ensembles are demoted to *ablations* that the significance
> analysis shows are within the seed-noise band. Figures are labeled accordingly.

In [ ]:
import os, re, json, ast, shutil, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

def find_repo_root():
    for c in [Path.cwd()] + list(Path.cwd().resolve().parents) + [Path("/workspace/Sarcasm_detection")]:
        if (c/"04_outputs").exists() or (c/"01_data").exists():
            return c.resolve()
    raise RuntimeError("repo root not found (need a dir containing 04_outputs/ or 01_data/).")

ROOT   = find_repo_root()
OUT    = ROOT/"04_outputs"
TABLES = OUT/"tables"; PRED = OUT/"predictions"; CONFIGS = OUT/"configs"
TEST_AWP = OUT/"test_awp"
FINAL  = OUT/"finalized_outputs"
FT, FF, FD = FINAL/"tables", FINAL/"figures", FINAL/"diagrams"
for p in (FT, FF, FD): p.mkdir(parents=True, exist_ok=True)
print("ROOT :", ROOT)
print("FINAL:", FINAL)

# ---- robust locators (search main dir, then test_awp, then anywhere under ROOT) ----
def _first(*cands):
    for c in cands:
        if c and Path(c).exists(): return Path(c)
    return None

def find_table(name):
    hit = _first(TABLES/name, TEST_AWP/"tables"/name, CONFIGS/name, TEST_AWP/"configs"/name)
    if hit: return hit
    for p in ROOT.rglob(name):
        if "finalized_outputs" not in p.parts: return p
    return None

def load_table(name):
    p = find_table(name)
    return pd.read_csv(p) if (p and p.suffix==".csv") else (json.load(open(p)) if p else None)

def find_pred(stem):
    n = f"{stem}_test_predictions.csv"
    hit = _first(PRED/n, TEST_AWP/"predictions"/n)
    if hit: return hit
    for p in ROOT.rglob(n):
        if "finalized_outputs" not in p.parts: return p
    return None

def find_raw_split(corpus, split):
    n = f"{corpus}_binary_{split}.csv"
    for p in ROOT.rglob(n):
        if "finalized_outputs" not in p.parts: return p
    return None

print("sanity — grand ranking found at:", find_table("15_grand_ranking.csv"))
print("sanity — 09b test preds found at:", find_pred("09b_fgm_awp_ben_sarc_binary"))

ROOT : /Users/sefayet/Desktop/Github/Sarcasm_detection
FINAL: /Users/sefayet/Desktop/Github/Sarcasm_detection/04_outputs/finalized_outputs
sanity — grand ranking found at: /Users/sefayet/Desktop/Github/Sarcasm_detection/04_outputs/tables/15_grand_ranking.csv
sanity — 09b test preds found at: /Users/sefayet/Desktop/Github/Sarcasm_detection/04_outputs/predictions/09b_fgm_awp_ben_sarc_binary_test_predictions.csv


## 1 · Figure engine  
One consistent, colourful, journal-grade style; one function per figure.

In [ ]:
import ast, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mp
warnings.filterwarnings("ignore")

# ---- restrained, print/grayscale-friendly palette ----
C = {
    "prior":    "#3B6EA5",   # prior-work / reproduced baselines (blue)
    "ours":     "#2E8B6F",   # this work (teal-green)
    "proposed": "#14513E",   # proposed (dark green)
    "reported": "#8C8C8C",   # reference (grey)
    "accent":   "#C77A30",   # FGM / highlight (ochre)
    "accent2":  "#A23B3B",   # AWP / significant (brick red)
    "nsig":     "#AEB6BD",   # not significant (grey)
    "grid":     "#DDDDDD",
}
SEQ = "viridis"; DIVERGE = "RdYlGn"
IN1, IN2 = 3.5, 7.16   # IEEE single / double column width (inches)

def set_style():
    plt.rcParams.update({
        "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
        "font.family": "serif",
        "font.serif": ["Times New Roman","Times","Nimbus Roman","DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 8.5, "axes.titlesize": 9.5, "axes.titleweight": "bold",
        "axes.labelsize": 8.5, "axes.linewidth": 0.8, "axes.edgecolor": "#333333",
        "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.2,
        "legend.frameon": True, "legend.framealpha": 0.92, "legend.edgecolor": "#CCCCCC",
        "axes.grid": False, "grid.color": C["grid"], "grid.linewidth": 0.6,
        "axes.axisbelow": True, "figure.facecolor": "white", "axes.facecolor": "white",
        "lines.linewidth": 1.4, "patch.linewidth": 0.6,
    })

def _save(fig, outdir, name):
    outdir = Path(outdir)
    for ext in ("pdf","png"):
        fig.savefig(outdir/f"{name}.{ext}")
    plt.close(fig); return f"{name}.png"

def cm_from_str(s): return np.array(ast.literal_eval(s)) if isinstance(s,str) else np.array(s)

# ---------------- human-readable names ----------------
PRETTY = {
 "11_stacking_LR":"Stacking ensemble","11_soft_vote_all":"Soft-vote ensemble",
 "11_hard_vote_all":"Hard-vote ensemble","11_soft_vote_top5":"Soft-vote (top-5)",
 "11_soft_vote_top3":"Soft-vote (top-3)",
 "09b_fgm_awp":"BanglaBERT + FGM+AWP (proposed)",
 "09_FINAL_proposed (seed-mean)":"Proposed (5-seed mean)","09_FINAL_proposed":"Proposed (5-seed mean)",
 "05_baseline_banglabert":"BanglaBERT (full FT)","06_banglabert":"BanglaBERT (full FT)",
 "06_muril":"MuRIL","06_xlm-roberta":"XLM-RoBERTa",
 "06_sagorsarker-bb":"BanglaBERT-Sagorsarker","06_mbert":"mBERT (full FT)",
 "07_fgm_awp":"FGM+AWP","07_fgm_e05":"FGM","07_freelb_k3":"FreeLB","07_awp":"AWP",
 "08_ZIHAN_repro":"Zihan recipe (clean)",
 "04_bengali-bert":"Bengali-BERT (frozen)","04_sagorsarker-bb":"Sagorsarker (frozen)",
 "04_mbert":"mBERT (frozen)",
 "03_LSTM_same_split":"LSTM","03_LSTM_CNN_same_split":"LSTM+CNN",
 "03_LSTM_CNN_GloVe_same_split":"LSTM+CNN+GloVe","03_StackedLSTM_CNN_GloVe_same_split":"Stacked LSTM+CNN+GloVe",
 "03_BiLSTM_CNN_GloVe_same_split":"BiLSTM+CNN+GloVe","03_BiLSTM_GloVe_same_split":"BiLSTM+GloVe",
 "02_classical_MultinomialNB":"Multinomial NB","02_classical_KernelSVM":"Kernel SVM",
 "02_classical_RandomForest":"Random Forest","02_classical_LogisticRegression":"Logistic Regression",
 "02_classical_LinearSVM":"Linear SVM","02_classical_DecisionTree":"Decision Tree","02_classical_KNN":"k-NN",
}
BACKBONE = {"banglabert":"BanglaBERT","muril":"MuRIL","xlm-roberta":"XLM-RoBERTa",
            "sagorsarker-bb":"Sagorsarker","mbert":"mBERT"}
def pretty(m):
    m=str(m)
    if m in PRETTY: return PRETTY[m]
    import re
    s=re.sub(r"^\d+[a-z]?_","",m).replace("classical_","").replace("_same_split","").replace("_"," ")
    return s

def _hlab(ax,bars,vals,dx=0.003,fs=7.5,fmt="{:.3f}"):
    for b,v in zip(bars,vals):
        ax.text(v+dx,b.get_y()+b.get_height()/2,fmt.format(v),ha="left",va="center",fontsize=fs)
def _vlab(ax,bars,vals,dy=0.0,fs=7.5,fmt="{:.3f}"):
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,v+dy,fmt.format(v),ha="center",va="bottom",fontsize=fs)

# ======================================================================
def fig_class_distribution(SS, outdir):
    ss=SS.copy(); ss["cp"]=ss["class_proportions"].apply(lambda x: json.loads(x) if isinstance(x,str) else x)
    specs=[("ben_sarc_binary","Ben-Sarc",["Non-Sarc","Sarc"]),
           ("banglasarc_binary","BanglaSarc",["Non-Sarc","Sarc"]),
           ("banglasarc3_ternary","BanglaSarc3",["Non-Sarc","Sarc","Neutral"])]
    cols=[C["prior"],C["accent2"],C["ours"]]
    fig,axes=plt.subplots(1,3,figsize=(IN2,2.5))
    for ax,(var,title,labels) in zip(axes,specs):
        sub=ss[ss["variant"]==var]; counts={}
        for _,r in sub.iterrows():
            for k,p in r["cp"].items(): counts[k]=counts.get(k,0)+p*r["n"]
        keys=sorted(counts,key=lambda k:int(k)); vals=[int(round(counts[k])) for k in keys]
        bars=ax.bar(labels[:len(vals)],vals,color=cols[:len(vals)],edgecolor="white",linewidth=0.8)
        for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,v,f"{v:,}",ha="center",va="bottom",fontsize=7)
        ax.set_title(title); ax.margins(y=0.18); ax.tick_params(labelsize=7)
        ax.spines[["top","right"]].set_visible(False)
    axes[0].set_ylabel("count")
    fig.tight_layout()
    return _save(fig,outdir,"F_class_distribution")

def fig_text_length(RAW,outdir):
    w=RAW["text"].astype(str).str.split().apply(len); med=int(w.median()); clip=w.clip(upper=100)
    fig,ax=plt.subplots(figsize=(IN1,2.4))
    ax.hist(clip,bins=40,color=C["prior"],edgecolor="white",linewidth=0.3)
    ax.axvline(med,color=C["accent2"],ls="--",lw=1.3,label=f"median = {med} words")
    ax.axvline(128,color=C["proposed"],ls=":",lw=1.3,label="128-token cap")
    ax.set_xlabel("words per comment (clipped at 100)"); ax.set_ylabel("count")
    ax.legend(); ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_text_length")

def fig_classical(CL,outdir):
    d=CL[CL["dataset"]=="ben_sarc_binary"].copy(); d["name"]=d["model"].map(lambda m: pretty(f"02_classical_{m}"))
    d=d.sort_values("test_macro_f1")
    fig,ax=plt.subplots(figsize=(IN1,2.6)); best=d["test_macro_f1"].max()
    colors=[C["accent"] if v==best else C["prior"] for v in d["test_macro_f1"]]
    bars=ax.barh(d["name"],d["test_macro_f1"],color=colors,edgecolor="white")
    _hlab(ax,bars,d["test_macro_f1"].values); ax.set_xlim(0,0.75); ax.set_xlabel("test macro-F1")
    ax.spines[["top","right"]].set_visible(False); ax.xaxis.grid(True)
    return _save(fig,outdir,"F_classical_ml")

def fig_dl_vs_reference(DL,outdir):
    d=DL.copy(); d["name"]=d["label"].map(lambda l: pretty(f"03_{l}_same_split")); d=d.sort_values("test_macro_f1")
    x=np.arange(len(d)); w=0.38
    fig,ax=plt.subplots(figsize=(IN2,2.7))
    ax.bar(x-w/2,d["test_macro_f1"],w,label="reproduced (clean split)",color=C["ours"],edgecolor="white")
    ax.bar(x+w/2,d["reference_f1"],w,label="Lora et al. (reported)",color=C["reported"],edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels(d["name"],rotation=18,ha="right",fontsize=7)
    ax.set_ylabel("macro-F1"); ax.set_ylim(0,0.86); ax.yaxis.grid(True)
    ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5,1.16), frameon=False)
    ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_dl_vs_reference")

def fig_repro_fidelity(RG,outdir):
    d=RG.copy(); d=d[d["task"]=="ben_sarc_binary"] if "task" in d.columns else d
    d["name"]=d["encoder"].map(lambda e:{"bengali-bert":"Bengali-BERT","sagorsarker-bb":"Sagorsarker","mbert":"mBERT"}.get(e,e))
    d=d.sort_values("test_macro_f1"); x=np.arange(len(d)); w=0.38
    fig,ax=plt.subplots(figsize=(IN1,2.5))
    ax.bar(x-w/2,d["test_macro_f1"],w,label="ours (clean)",color=C["ours"],edgecolor="white")
    ax.bar(x+w/2,d["reference_f1"],w,label="Lora reported",color=C["reported"],edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels(d["name"],fontsize=7.5); ax.set_ylabel("macro-F1"); ax.set_ylim(0,0.82)
    ax.legend(); ax.yaxis.grid(True); ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_repro_fidelity")

def fig_multibackbone(PIVOT,outdir):
    d=PIVOT.copy(); d["backbone"]=d["backbone"].map(lambda b:BACKBONE.get(b,b)); d=d.set_index("backbone")
    order=["ben_sarc_binary","banglasarc_binary","banglasarc3_binary","banglasarc3_ternary","mean"]
    lbl=["Ben-Sarc","BanglaSarc","BanglaSarc3\n(bin)","BanglaSarc3\n(tern)","mean"]
    d=d[[c for c in order if c in d.columns]].sort_values("mean",ascending=False)
    fig,ax=plt.subplots(figsize=(IN2,2.9))
    im=ax.imshow(d.values,cmap=SEQ,aspect="auto",vmin=0.55,vmax=1.0)
    ax.set_xticks(range(len(d.columns))); ax.set_xticklabels(lbl[:len(d.columns)],fontsize=7.5)
    ax.set_yticks(range(len(d.index))); ax.set_yticklabels(d.index,fontsize=8)
    for i in range(d.shape[0]):
        for j in range(d.shape[1]):
            v=d.values[i,j]; ax.text(j,i,f"{v:.3f}",ha="center",va="center",
                                     color="white" if v<0.8 else "black",fontsize=7.5)
    cb=fig.colorbar(im,ax=ax,fraction=0.046,pad=0.03); cb.set_label("macro-F1",fontsize=8); cb.ax.tick_params(labelsize=7)
    return _save(fig,outdir,"F_multibackbone")

def fig_adversarial(AP,outdir,ref_value=None,ref_label="full fine-tuning (no adversarial)"):
    d=AP.set_index("config"); order=["fgm_awp","fgm_e05","freelb_k3","awp"]
    nice={"fgm_awp":"FGM+AWP","fgm_e05":"FGM","freelb_k3":"FreeLB","awp":"AWP"}
    d=d.reindex([o for o in order if o in d.index]); vals=d["ben_sarc_binary"]
    fig,ax=plt.subplots(figsize=(IN1,2.6))
    bars=ax.bar([nice[i] for i in d.index],vals,color=C["nsig"],edgecolor="white",width=0.62)
    _vlab(ax,bars,vals.values,dy=0.0004)
    lo=min(vals.min(), ref_value if ref_value else vals.min())
    if ref_value is not None:
        ax.axhline(ref_value,color=C["ours_dk"] if "ours_dk" in C else C["proposed"],ls="--",lw=1.4,
                   label=f"{ref_label} = {ref_value:.3f}")
        ax.legend(loc="upper center", bbox_to_anchor=(0.5,1.18), frameon=False)
    ax.set_ylabel("test macro-F1"); ax.set_ylim(0.785, max(0.806, (ref_value or 0)+0.002)); ax.yaxis.grid(True)
    ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_adversarial")

def fig_loss_curve(L,outdir):
    d=L.copy(); fig,(a1,a2)=plt.subplots(1,2,figsize=(IN2,2.5))
    a1.plot(d["epoch"],d["train_loss"],"-o",color=C["prior"],label="train",ms=4)
    a1.plot(d["epoch"],d["val_loss"],"-s",color=C["accent"],label="validation",ms=4)
    a1.set_xlabel("epoch"); a1.set_ylabel("loss"); a1.set_title("Loss"); a1.legend(); a1.grid(True)
    a2.plot(d["epoch"],d["val_macro_f1"],"-^",color=C["ours"],ms=5)
    bst=d.loc[d["val_macro_f1"].idxmax()]
    a2.scatter([bst["epoch"]],[bst["val_macro_f1"]],s=90,facecolor="none",edgecolor=C["accent2"],lw=1.6,zorder=5,label="selected")
    a2.set_xlabel("epoch"); a2.set_ylabel("macro-F1"); a2.set_title("Validation macro-F1"); a2.legend(); a2.grid(True)
    for a in (a1,a2): a.spines[["top","right"]].set_visible(False)
    fig.suptitle("Proposed: BanglaBERT full FT + label-smoothed head", fontsize=10, y=1.04)
    fig.tight_layout()
    return _save(fig,outdir,"F_proposed_loss_curve")

def fig_epsilon(E,outdir):
    d=E.sort_values("eps"); fig,ax=plt.subplots(figsize=(IN1,2.3))
    ax.plot(d["eps"],d["val_macro_f1"],"-o",color=C["ours"],label="validation",ms=4)
    ax.plot(d["eps"],d["test_macro_f1"],"-s",color=C["accent"],label="test",ms=4)
    b=d.loc[d["val_macro_f1"].idxmax(),"eps"]; ax.axvline(b,color=C["accent2"],ls="--",lw=1.1,label=f"selected $\\epsilon$={b}")
    ax.set_xlabel("FGM $\\epsilon$"); ax.set_ylabel("macro-F1"); ax.legend(); ax.grid(True)
    ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_epsilon_sweep")

def fig_seed_stability(MS,outdir):
    d=MS.copy(); mean=d["test_macro_f1"].mean(); std=d["test_macro_f1"].std()
    fig,ax=plt.subplots(figsize=(IN1,2.4)); x=range(len(d))
    ax.axhspan(mean-std,mean+std,color=C["ours"],alpha=0.15,label=f"mean $\\pm$ std = {mean:.4f} $\\pm$ {std:.4f}")
    ax.axhline(mean,color=C["proposed"],ls="--",lw=1.1)
    ax.plot(x,d["test_macro_f1"],"o",color=C["ours"],ms=8,label="per-seed test macro-F1")
    ax.set_xticks(list(x)); ax.set_xticklabels([f"s{sd}" for sd in d["seed"]])
    ax.set_ylabel("test macro-F1"); ax.grid(True)
    ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5,1.20), frameon=False)
    ax.margins(x=0.10)
    ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_seed_stability")

def fig_cross_dataset(PIV,outdir):
    d=PIV.set_index("source"); order=["ben_sarc_binary","banglasarc_binary","banglasarc3_binary"]
    lbl={"ben_sarc_binary":"Ben-Sarc","banglasarc_binary":"BanglaSarc","banglasarc3_binary":"BanglaSarc3"}
    d=d.reindex(index=[o for o in order if o in d.index],columns=[o for o in order if o in d.columns])
    fig,ax=plt.subplots(figsize=(IN1,3.1))
    im=ax.imshow(d.values,cmap=DIVERGE,vmin=0.3,vmax=1.0,aspect="auto")
    ax.set_xticks(range(len(d.columns))); ax.set_xticklabels([lbl[c] for c in d.columns],fontsize=7.5)
    ax.set_yticks(range(len(d.index))); ax.set_yticklabels([lbl[i] for i in d.index],fontsize=7.5)
    ax.set_xlabel("evaluated on"); ax.set_ylabel("trained on")
    for i in range(d.shape[0]):
        for j in range(d.shape[1]):
            ax.text(j,i,f"{d.values[i,j]:.3f}",ha="center",va="center",fontsize=8.5,fontweight="bold")
            if d.index[i]==d.columns[j]: ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,fill=False,edgecolor="#111",lw=2))
    cb=fig.colorbar(im,ax=ax,fraction=0.046,pad=0.03); cb.set_label("macro-F1",fontsize=8); cb.ax.tick_params(labelsize=7)
    return _save(fig,outdir,"F_cross_dataset_heatmap")

def fig_indomain_vs_transfer(M,outdir):
    lbl={"ben_sarc_binary":"Ben-Sarc","banglasarc_binary":"BanglaSarc","banglasarc3_binary":"BanglaSarc3"}
    rows=[]
    for src in M["source"].unique():
        s=M[M["source"]==src]; rows.append((lbl.get(src,src),
            s[s["in_domain"]==True]["macro_f1"].mean(), s[s["in_domain"]==False]["macro_f1"].mean()))
    r=pd.DataFrame(rows,columns=["src","ind","off"]).sort_values("ind",ascending=False)
    x=np.arange(len(r)); w=0.38
    fig,ax=plt.subplots(figsize=(IN1,2.6))
    ax.bar(x-w/2,r["ind"],w,label="in-domain",color=C["ours"],edgecolor="white")
    ax.bar(x+w/2,r["off"],w,label="cross-corpus (zero-shot)",color=C["accent2"],edgecolor="white")
    for i in range(len(r)):
        ax.text(i,r["ind"].iloc[i]+0.03,f"$-${r['ind'].iloc[i]-r['off'].iloc[i]:.2f}",ha="center",color=C["accent2"],fontsize=7.5,fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(r["src"],fontsize=7.5); ax.set_ylabel("macro-F1"); ax.set_ylim(0,1.12)
    ax.legend(loc="upper right"); ax.yaxis.grid(True); ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_indomain_vs_transfer")

def fig_reliability(preds,outdir,nbins=15):
    fig,ax=plt.subplots(figsize=(IN1,3.0))
    ax.plot([0,1],[0,1],"--",color="#999",lw=1.1,label="perfect")
    sty={"proposed":(C["ours"],"o"),"baseline":(C["accent"],"s")}
    for name,df in preds.items():
        conf=df[["prob_0","prob_1"]].max(axis=1).values; corr=df["correct"].values.astype(float)
        b=np.linspace(0,1,nbins+1); idx=np.clip(np.digitize(conf,b)-1,0,nbins-1)
        xs,ys,ece=[],[],0.0
        for k in range(nbins):
            m=idx==k
            if m.sum()>0:
                xs.append(conf[m].mean()); ys.append(corr[m].mean()); ece+=(m.sum()/len(conf))*abs(corr[m].mean()-conf[m].mean())
        key="proposed" if "proposed" in name.lower() else "baseline"; col,mk=sty[key]
        ax.plot(xs,ys,marker=mk,color=col,ms=5,label=f"{name} (ECE={ece:.3f})")
    ax.legend(loc="upper left", fontsize=6.8)
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_xlim(0.45,1.02); ax.set_ylim(0.45,1.02)
    ax.grid(True); ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_reliability")

def fig_calibration_bars(CAL,outdir):
    d=CAL.copy(); nm={"09b_fgm_awp":"Proposed\n(+ LS head)","05_baseline_banglabert":"Full FT\n(no LS)"}
    d["disp"]=d["model"].map(lambda x:nm.get(x,x)); x=np.arange(len(d)); w=0.38
    fig,ax=plt.subplots(figsize=(IN1,2.4))
    b1=ax.bar(x-w/2,d["ece"],w,label="ECE (raw)",color=C["accent2"],edgecolor="white")
    b2=ax.bar(x+w/2,d["ece_after"],w,label="ECE (T-scaled)",color=C["ours"],edgecolor="white")
    _vlab(ax,b1,d["ece"].values,dy=0.0005); _vlab(ax,b2,d["ece_after"].values,dy=0.0005)
    ax.set_xticks(x); ax.set_xticklabels(d["disp"]); ax.set_ylabel("Expected Calibration Error")
    ax.legend(); ax.yaxis.grid(True); ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_calibration")

def fig_confusion(cm,outdir,title,name,labels=("Non-Sarc","Sarc")):
    cm=np.array(cm); fig,ax=plt.subplots(figsize=(IN1*0.8,2.5)); im=ax.imshow(cm,cmap="Greens")
    ax.set_xticks([0,1]); ax.set_xticklabels(labels); ax.set_yticks([0,1]); ax.set_yticklabels(labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("actual"); tot=cm.sum()
    for i in range(2):
        for j in range(2):
            ax.text(j,i,f"{cm[i,j]:,}\n({cm[i,j]/tot*100:.1f}%)",ha="center",va="center",
                    fontsize=9,color="white" if cm[i,j]>cm.max()*0.6 else "#111")
    return _save(fig,outdir,name)

def fig_length_accuracy(LS,outdir):
    d=LS.copy(); fig,ax=plt.subplots(figsize=(IN1,2.4))
    norm=plt.Normalize(d["acc"].min()-0.01,d["acc"].max()+0.01); colors=plt.cm.viridis(norm(d["acc"]))
    bars=ax.bar(d["bucket"],d["acc"],color=colors,edgecolor="white")
    for b,v,n in zip(bars,d["acc"],d["n"]):
        ax.text(b.get_x()+b.get_width()/2,v+0.004,f"{v:.3f}",ha="center",fontsize=7)
        ax.text(b.get_x()+b.get_width()/2,0.515,f"n={n}",ha="center",fontsize=6.5,color="white")
    ax.set_ylim(0.5,0.9); ax.set_xlabel("comment length (chars)"); ax.set_ylabel("accuracy")
    ax.tick_params(axis="x",labelsize=7); ax.yaxis.grid(True); ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_length_accuracy")

def fig_error_asymmetry(cm,outdir):
    cm=np.array(cm); fp=cm[0,1]; fn=cm[1,0]
    fig,ax=plt.subplots(figsize=(IN1*0.85,2.3))
    bars=ax.bar(["False Positive\n(non-sarc→sarc)","False Negative\n(sarc→non-sarc)"],[fp,fn],
                color=[C["reported"],C["accent2"]],edgecolor="white",width=0.6)
    _vlab(ax,bars,[fp,fn],dy=2,fmt="{:.0f}")
    ax.set_ylabel("errors"); ax.tick_params(axis="x",labelsize=7)
    ax.yaxis.grid(True); ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_error_asymmetry")

def fig_headtohead(H,outdir):
    order=["REPORTED","REPRODUCED","This work"]
    def key(s):
        for i,k in enumerate(order):
            if k.lower() in s.lower(): return i
        return 9
    d=H.copy(); d["k"]=d["system"].map(key); d=d.sort_values("k")
    labels=["Lora et al.\n(reported)","Lora et al.\n(reproduced)","This work\n(best)"]
    cols=[C["reported"],C["prior"],C["ours"]]
    fig,ax=plt.subplots(figsize=(IN1,2.5))
    bars=ax.bar(labels[:len(d)],d["macro_f1"],color=cols[:len(d)],edgecolor="white",width=0.6)
    _vlab(ax,bars,d["macro_f1"].values,dy=0.006)
    ax.set_ylim(0,0.9); ax.set_ylabel("macro-F1"); ax.tick_params(axis="x",labelsize=7.5)
    ax.yaxis.grid(True); ax.spines[["top","right"]].set_visible(False)
    return _save(fig,outdir,"F_head_to_head")

# ---- CURATED best-per-stage ladder (proposed = BanglaBERT full FT + LS head) ----
def fig_ranking_ladder(RANK,H2H,outdir):
    import matplotlib.patches as mp
    r=RANK.copy()
    def best(prefixes):
        d=r[r["method"].astype(str).str.startswith(tuple(prefixes))]
        return d.sort_values("test_macro_f1",ascending=False).iloc[0] if len(d) else None
    lora_reported=None
    if H2H is not None:
        m=H2H[H2H["system"].str.contains("REPORTED",case=False,na=False)]
        if len(m): lora_reported=float(m["macro_f1"].iloc[0])
    stages=[]
    stages.append(("Best classical ML",best(["02_"]),"prior"))
    stages.append(("Best deep learning",best(["03_"]),"prior"))
    stages.append(("Best frozen transfer\n(Lora reproduced)",best(["04_"]),"prior"))
    b=r[r["method"].eq("09b_fgm_awp")]; b=b.iloc[0] if len(b) else best(["05_","06_"])
    stages.append(("Proposed\n(BanglaBERT full-FT\n+ LS head)",b,"proposed"))
    rows=[(lbl,float(x["test_macro_f1"]),grp) for lbl,x,grp in stages if x is not None]
    d=pd.DataFrame(rows,columns=["label","f1","grp"])
    cmap={"prior":C["prior"],"proposed":C["proposed"]}
    fig,ax=plt.subplots(figsize=(IN1*1.16, 3.1))
    bars=ax.barh(d["label"],d["f1"],color=[cmap[g] for g in d["grp"]],edgecolor="white")
    _hlab(ax,bars,d["f1"].values,dx=0.008)
    handles=[mp.Patch(color=C["prior"],label="Prior work / reproduced"),
             mp.Patch(color=C["proposed"],label="Proposed")]
    if lora_reported is not None:
        ax.axvline(lora_reported,color=C["reported"],ls="--",lw=1.2)
        handles.append(plt.Line2D([0],[0],color=C["reported"],ls="--",lw=1.2,
                                  label=f"Lora et al. reported ({lora_reported:.3f})"))
    ax.set_xlim(0,0.92); ax.set_xlabel("test macro-F1"); ax.xaxis.grid(True)
    ax.spines[["top","right"]].set_visible(False)
    ax.legend(handles=handles,loc="lower right",fontsize=6.4,framealpha=0.95)
    return _save(fig,outdir,"F_grand_ranking")

# ---- CURATED significance figure (key comparators, real names) ----
def fig_significance(SIG,outdir):
    if SIG is None: return None
    keep=["04_bengali-bert","04_mbert","06_mbert","06_sagorsarker-bb","06_xlm-roberta","06_muril","05_baseline_banglabert","07_fgm_awp"]
    s=SIG[SIG["comparator"].isin(keep)].copy()
    s["name"]=s["comparator"].map(lambda m: pretty(m)+(" (frozen)" if m.startswith("04_") else ""))
    # fix frozen dup naming
    s["name"]=s.apply(lambda r:{"04_bengali-bert":"Bengali-BERT (frozen)","04_mbert":"mBERT (frozen)"}.get(r["comparator"],r["name"]),axis=1)
    s=s.sort_values("f1_comparator")
    fig,ax=plt.subplots(figsize=(IN1,2.9))
    colors=[C["accent2"] if bool(v) else C["nsig"] for v in s["sig_0.05"]]
    bars=ax.barh(s["name"],s["f1_comparator"],color=colors,edgecolor="white")
    prop=float(s["f1_proposed"].iloc[0]); ax.axvline(prop,color=C["proposed"],ls="--",lw=1.3,label=f"proposed = {prop:.3f}")
    # (proposed = BanglaBERT full FT + label-smoothed head)
    _hlab(ax,bars,s["f1_comparator"].values,dx=0.003)
    ax.set_xlim(0.6,0.83); ax.set_xlabel("macro-F1"); ax.xaxis.grid(True)
    ax.spines[["top","right"]].set_visible(False)
    handles=[mp.Patch(color=C["accent2"],label="proposed sig. better ($p<0.05$)"),
             mp.Patch(color=C["nsig"],label="not separable"),
             plt.Line2D([0],[0],color=C["proposed"],ls="--",label=f"proposed = {prop:.3f}")]
    ax.legend(handles=handles,loc="lower right",fontsize=6.6)
    fig.tight_layout()
    return _save(fig,outdir,"F_significance")

## 2 · Generate all figures  →  `finalized_outputs/figures/`

In [ ]:
set_style()
made, skipped = [], []

def _try(fn, *a, **k):
    try:
        r = fn(*a, **k); made.append(r); print("  ok:", r)
    except Exception as e:
        skipped.append((getattr(fn,'__name__','?'), str(e))); print("  SKIP", fn.__name__, "->", e)

# dataset foundation
ss = load_table("01_split_summary.csv")
_try(fig_class_distribution, ss, FF)
raw_parts = [find_raw_split("ben_sarc", s) for s in ("train","val","test")]
if all(raw_parts):
    raw = pd.concat([pd.read_csv(p) for p in raw_parts], ignore_index=True)
    _try(fig_text_length, raw, FF)
else:
    skipped.append(("fig_text_length","raw ben_sarc splits not found")); print("  SKIP text_length (raw splits missing)")

# reproduction
_try(fig_classical, load_table("02_classical_ml_results.csv"), FF)
_try(fig_dl_vs_reference, load_table("03_dl_results.csv"), FF)
_try(fig_repro_fidelity, load_table("04_reference_gap.csv"), FF)

# this-work models
_try(fig_multibackbone, load_table("06_macro_f1_pivot.csv"), FF)
# vanilla full-FT macro-F1 as the reference line for the adversarial ablation
vanilla_ref = None
try:
    from sklearn.metrics import f1_score as _f1
    _vp = pd.read_csv(find_pred("05_baseline_banglabert_ben_sarc_binary"))
    vanilla_ref = round(_f1(_vp["gold_label"], _vp["pred_label"], average="macro"), 4)
except Exception as _e:
    print("  (vanilla ref unavailable:", _e, ")")
_try(fig_adversarial, load_table("07_adversarial_macro_f1_pivot.csv"), FF, ref_value=vanilla_ref)
_try(fig_loss_curve, load_table("09b_fgm_awp_loss_curve.csv"), FF)
_try(fig_epsilon, load_table("09_epsilon_sweep.csv"), FF)
_try(fig_seed_stability, load_table("09_final_multiseed.csv"), FF)

# evaluation suite (the novelty)
_try(fig_cross_dataset, load_table("10_cross_dataset_pivot.csv"), FF)
_try(fig_indomain_vs_transfer, load_table("10_cross_dataset_matrix.csv"), FF)

p09b, p05 = find_pred("09b_fgm_awp_ben_sarc_binary"), find_pred("05_baseline_banglabert_ben_sarc_binary")
if p09b and p05:
    preds = {"Proposed (full FT + LS head)": pd.read_csv(p09b), "Full FT (no LS)": pd.read_csv(p05)}
    _try(fig_reliability, preds, FF)
else:
    skipped.append(("fig_reliability","prediction files missing")); print("  SKIP reliability")
_try(fig_calibration_bars, load_table("13_calibration.csv"), FF)

cm09b = cm_from_str(load_table("09b_fgm_awp_summary.csv")["confusion_matrix"].iloc[0])
_try(fig_confusion, cm09b, FF, "Confusion matrix — proposed model", "F_confusion_proposed")
_try(fig_length_accuracy, load_table("14_length_stratified.csv"), FF)
_try(fig_error_asymmetry, cm09b, FF)
_try(fig_headtohead, load_table("15_ours_vs_lora.csv"), FF)

rank, sig = load_table("15_grand_ranking.csv"), load_table("12_significance.csv")
_try(fig_ranking_ladder, rank, load_table("15_ours_vs_lora.csv"), FF)
_try(fig_significance, sig, FF)

print(f"\nGENERATED {len(made)} figures; skipped {len(skipped)}.")

  ok: F_class_distribution.png
  ok: F_text_length.png
  ok: F_classical_ml.png
  ok: F_dl_vs_reference.png
  ok: F_repro_fidelity.png
  ok: F_multibackbone.png
  ok: F_adversarial.png
  ok: F_proposed_loss_curve.png
  ok: F_epsilon_sweep.png
  ok: F_seed_stability.png
  ok: F_cross_dataset_heatmap.png
  ok: F_indomain_vs_transfer.png
  ok: F_reliability.png
  ok: F_calibration.png
  ok: F_confusion_proposed.png
  ok: F_length_accuracy.png
  ok: F_error_asymmetry.png
  ok: F_head_to_head.png
  ok: F_grand_ranking.png
  ok: F_significance.png

GENERATED 20 figures; skipped 0.


## 3 · Optional CPU cross-corpus classical baseline (TF-IDF + Logistic Regression)

Shows the cross-corpus collapse is **not** a transformer artefact — a classical model also fails
to transfer. Pure CPU, seconds to run. Skips gracefully if the other corpora's splits aren't found.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

def _lab_col(df):
    for c in ("label_binary","label","gold_label"):
        if c in df.columns: return c
    return df.columns[1]

CORPORA = ["ben_sarc","banglasarc","banglasarc3"]
splits = {c: (find_raw_split(c,"train"), find_raw_split(c,"test")) for c in CORPORA}
avail  = [c for c in CORPORA if all(splits[c])]
print("corpora available for cross-corpus classical:", avail)

if len(avail) >= 2:
    rows=[]
    for src in avail:
        tr = pd.read_csv(splits[src][0])
        vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000)
        Xtr = vec.fit_transform(tr["text"].astype(str)); ytr = tr[_lab_col(tr)].astype(int)
        clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
        for tgt in avail:
            te = pd.read_csv(splits[tgt][1])
            Xte = vec.transform(te["text"].astype(str)); yte = te[_lab_col(te)].astype(int)
            yp = clf.predict(Xte)
            rows.append({"source":src,"target":tgt,"in_domain":src==tgt,
                         "macro_f1":round(f1_score(yte,yp,average='macro'),4),
                         "accuracy":round(accuracy_score(yte,yp),4)})
    ccc = pd.DataFrame(rows)
    ccc.to_csv(FT/"17_cross_corpus_classical_matrix.csv", index=False)
    piv = ccc.pivot(index="source", columns="target", values="macro_f1")
    piv.to_csv(FT/"17_cross_corpus_classical_pivot.csv")
    print(ccc.to_string(index=False))
    # figure: classical transfer heatmap (reuse cross-dataset styling)
    import matplotlib.pyplot as plt
    lbl={"ben_sarc":"Ben-Sarc","banglasarc":"BanglaSarc","banglasarc3":"BanglaSarc3"}
    d=piv.reindex(index=[c for c in CORPORA if c in piv.index], columns=[c for c in CORPORA if c in piv.columns])
    fig,ax=plt.subplots(figsize=(6.6,5.6)); im=ax.imshow(d.values,cmap=DIVERGE,vmin=0.3,vmax=1.0)
    ax.set_xticks(range(len(d.columns))); ax.set_xticklabels([lbl[c] for c in d.columns])
    ax.set_yticks(range(len(d.index)));  ax.set_yticklabels([lbl[i] for i in d.index])
    ax.set_xlabel("evaluated on →"); ax.set_ylabel("trained on ↓")
    for i in range(d.shape[0]):
        for j in range(d.shape[1]):
            ax.text(j,i,f"{d.values[i,j]:.3f}",ha="center",va="center",fontweight="bold")
            if d.index[i]==d.columns[j]: ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,fill=False,edgecolor="#111",lw=2.5))
    ax.set_title("Cross-corpus transfer — TF-IDF + LogReg (classical control)"); ax.grid(False)
    fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04,label="macro-F1")
    for ext in ("png","pdf"): fig.savefig(FF/f"F_cross_corpus_classical.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig); print("saved F_cross_corpus_classical.png + 17_cross_corpus_classical_*.csv")
else:
    print("SKIPPED: need >=2 corpora splits present in the repo. "
          "Ensure {corpus}_binary_train.csv / _test.csv exist for banglasarc and banglasarc3.")

corpora available for cross-corpus classical: ['ben_sarc', 'banglasarc', 'banglasarc3']
     source      target  in_domain  macro_f1  accuracy
   ben_sarc    ben_sarc       True    0.6562    0.6563
   ben_sarc  banglasarc      False    0.4281    0.4935
   ben_sarc banglasarc3      False    0.6122    0.6157
 banglasarc    ben_sarc      False    0.3663    0.4959
 banglasarc  banglasarc       True    0.8804    0.8922
 banglasarc banglasarc3      False    0.3969    0.5032
banglasarc3    ben_sarc      False    0.5891    0.5892
banglasarc3  banglasarc      False    0.4466    0.4806
banglasarc3 banglasarc3       True    0.7067    0.7067
saved F_cross_corpus_classical.png + 17_cross_corpus_classical_*.csv


## 4 · Copy the four schematic diagrams  →  `finalized_outputs/diagrams/`

In [ ]:
DIAGRAMS = ["F1_Methodology_overview","F2_Proposed_Method","F3_Adversial_step","F4_Data_Leakage_Control"]
def _find_diagram(stem, ext):
    n=f"{stem}.{ext}"
    for p in ROOT.rglob(n):
        if p.parent != FD:   # skip the destination itself
            return p
    return (FD/n) if (FD/n).exists() else None

dcopied, dmissing = [], []
for stem in DIAGRAMS:
    for ext in ("png","svg"):
        src=_find_diagram(stem, ext)
        if src and src.parent != FD:
            shutil.copy2(src, FD/f"{stem}.{ext}"); dcopied.append(f"{stem}.{ext}")
        elif (FD/f"{stem}.{ext}").exists():
            dcopied.append(f"{stem}.{ext} (already in place)")
        else:
            dmissing.append(f"{stem}.{ext}")
print("diagrams present:", *dcopied, sep="\n  ")
if dmissing: print("MISSING diagrams:", dmissing)

diagrams present:
  F1_Methodology_overview.png
  F2_Proposed_Method.png
  F3_Adversial_step.png
  F4_Data_Leakage_Control.png
MISSING diagrams: ['F1_Methodology_overview.svg', 'F2_Proposed_Method.svg', 'F3_Adversial_step.svg', 'F4_Data_Leakage_Control.svg']


## 5 · Copy paper tables  →  `finalized_outputs/tables/`

In [ ]:
TABLE_WHITELIST = [
    "01_split_summary.csv","01_dedup_report.csv","01_cross_corpus_overlap.csv",
    "02_classical_ml_results.csv","03_dl_results.csv","03_dl_reference_gap_same_split.csv",
    "04_frozen_transfer_results.csv","04_reference_gap.csv",
    "05_baseline_banglabert_summary.csv","06_multi_backbone_summary.csv","06_macro_f1_pivot.csv",
    "07_adversarial_summary.csv","07_adversarial_macro_f1_pivot.csv","07_best_adversarial_per_task.csv",
    "08_pipeline_search.csv","09_epsilon_sweep.csv","09_final_multiseed.csv",
    "09b_fgm_awp_summary.csv","09b_fgm_awp_multiseed.csv","09b_fgm_awp_config.json",
    "10_cross_dataset_matrix.csv","10_cross_dataset_pivot.csv",
    "11_ensemble_results.csv","12_significance.csv","12_ensemble_significance.csv",
    "13_calibration.csv","14_length_stratified.csv","14_high_conf_errors.csv",
    "15_grand_ranking.csv","15_ours_vs_lora.csv","15_ours_vs_lora_key_methods.csv","15_zihan_leakage_aside.csv",
]
tcopied, tmissing = [], []
for n in TABLE_WHITELIST:
    src = find_table(n)
    if src: shutil.copy2(src, FT/n); tcopied.append(n)
    else:   tmissing.append(n)
print(f"copied {len(tcopied)} tables; missing {len(tmissing)}: {tmissing}")

copied 32 tables; missing 0: []


## 6 · Headline table + manifest

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
def _metrics(stem):
    p=find_pred(stem)
    if not p: return None
    d=pd.read_csv(p); g=d["gold_label"] if "gold_label" in d else d["y_true"]; y=d["pred_label"] if "pred_label" in d else d["y_pred"]
    return round(f1_score(g,y,average="macro"),4), round(accuracy_score(g,y),4)

ens = load_table("11_ensemble_results.csv")
best_ens = ens.sort_values("test_macro_f1",ascending=False).iloc[0] if ens is not None else None
h2h = load_table("15_ours_vs_lora.csv")
head=[]
if best_ens is not None:
    head.append({"role":"best overall (ensemble)","system":f"ensemble:{best_ens['ensemble']}","macro_f1":round(float(best_ens['test_macro_f1']),4),"accuracy":""})
for stem,role,label in [("09b_fgm_awp_ben_sarc_binary","proposed single model (deployable)","09b_fgm_awp (tuned FGM+AWP)"),
                        ("05_baseline_banglabert_ben_sarc_binary","strong single-model baseline","05 vanilla BanglaBERT full-FT")]:
    m=_metrics(stem)
    if m: head.append({"role":role,"system":label,"macro_f1":m[0],"accuracy":m[1]})
if h2h is not None:
    for _,r in h2h.iterrows():
        if "REPORTED" in str(r["system"]):   head.append({"role":"reference (Lora reported)","system":r["system"],"macro_f1":round(float(r["macro_f1"]),4),"accuracy":""})
        elif "REPRODUCED" in str(r["system"]):head.append({"role":"reference (Lora reproduced)","system":r["system"],"macro_f1":round(float(r["macro_f1"]),4),"accuracy":""})
hl=pd.DataFrame(head); hl.to_csv(FT/"16_headline_results.csv",index=False)
print(hl.to_string(index=False))

manifest={"root":str(ROOT),"generated_figures":sorted(p.name for p in FF.glob('*.png')),
          "vector_figures":sorted(p.name for p in FF.glob('*.pdf')),
          "tables":sorted(p.name for p in FT.iterdir()),
          "diagrams":sorted(p.name for p in FD.iterdir()),
          "n_figures":len(list(FF.glob('*.png'))),"n_tables":len(list(FT.glob('*.csv')))+len(list(FT.glob('*.json'))),
          "proposed_single_model":"09b_fgm_awp","best_overall":(str(best_ens['ensemble']) if best_ens is not None else None),
          "figures_skipped":skipped,"tables_missing":tmissing,"diagrams_missing":dmissing}
json.dump(manifest, open(FINAL/"MANIFEST.json","w"), indent=2)
print("\nMANIFEST written."); print(f"figures={manifest['n_figures']}  tables={manifest['n_tables']}  diagrams={len(manifest['diagrams'])}")

                              role                                          system  macro_f1 accuracy
           best overall (ensemble)                            ensemble:stacking_LR    0.8115         
proposed single model (deployable)                     09b_fgm_awp (tuned FGM+AWP)    0.8038   0.8041
      strong single-model baseline                   05 vanilla BanglaBERT full-FT    0.8025    0.803
         reference (Lora reported)           Lora et al. — best REPORTED (Table 9)    0.7547         
       reference (Lora reproduced) Lora et al. — best REPRODUCED (04_bengali-bert)    0.7444         

MANIFEST written.
figures=21  tables=35  diagrams=4


## Diagram note

`F1_Methodology_overview` and `F2_Proposed_Method` were copied as-is but still encode the **old
headline** ("Adversarially Robust Extension", "AR-BanglaBERT 0.804 … +5.7 pts over prior best").
Under the repositioned story (cross-corpus generalization + leakage-controlled rigor as the
contribution, adversarial training demoted to an ablation), **F1 must be redrawn** and **F2
retitled**. `F3` (adversarial step) and `F4` (data-leakage control) are fine to keep. Do this before
building the paper draft.